# HydroLand — Run Guide

**Collection ID:** `EO.UFZ.DAT.HYDROLAND`
**Responsable:** Jeisson Leal (UFZ)

Land Surface Hydrology Simulations from the Mesoscale Hydrologic Model (mHM/mRM):
river discharge, soil moisture, plus decadal indicators (aridity index, discharge
percentiles). Historical (1990–2014) and projection (2015–2049) periods at 5 km
resolution, 3 Climate DT models.

This notebook runs the **whole pipeline against real HydroLand data** — the actual
historical mHM source on disk — into a throwaway sandbox directory, and stops
**before** the upload step. No credentials needed, nothing is pushed to S3.

For the conda environment and `.env` credential setup (only needed for the real
upload step, not for this notebook), see the top-level
[`README.md`](../../README.md). For how HydroLand compares to the other three apps,
see [`apps/RUN_GUIDE.md`](../RUN_GUIDE.md).

Prefer a plain shell script over a notebook? See [`run_example.sh`](run_example.sh)
for the same three steps with no narration, just commands and a `SOURCE`/`TARGET`
pair to edit at the top.

## Files in this app

| File | Purpose |
|---|---|
| `catalog_config.yaml` | ALL HydroLand input: collection metadata, paths, simulation keys |
| `staging/stage.py` | Builds the catalog skeleton from config: `collection.json` + `data/YYYY/MM/<item>/` folders for every simulation × month × data_type |
| `staging/move.py` | Links/copies/moves source NetCDF files into the staged `YYYY/MM/item/` folders |
| `staging/update.py` | Generates STAC item metadata for the staged collection |
| `staging/delete_s3_collection_prefix.py` | Maintenance utility to delete a collection prefix from the S3 bucket |
| `staging/check_data.sh` | Optional pre-staging QC: checks one file per month is present, flags unrecognized filenames |
| `run_pipeline.sh` | Wraps `stage.py` + `update.py` + upload into one command (does **not** run `move.py` — it needs explicit per-source metadata flags) |

In [1]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import yaml

APP_DIR = Path.cwd()
assert (APP_DIR / "catalog_config.yaml").exists(), "run this notebook from apps/hydroland/"

COLLECTION_ID = "EO.UFZ.DAT.HYDROLAND"

# A throwaway sandbox so this notebook never touches the real production catalog.
SANDBOX = Path(tempfile.mkdtemp(prefix="hydroland_notebook_"))
ENV = {**os.environ, "CATALOG_HOME_DIR": str(SANDBOX)}
print(f"Sandbox: {SANDBOX}")

Sandbox: /tmp/hydroland_notebook_qy73optw


## 1. The config file

`catalog_config.yaml` is the single source of truth: collection metadata (title,
description, license, DOI), the simulations to publish, and the two HPC paths
(`app_data_dir`, `catalog_home_dir`). Both paths can be overridden per machine with
the `APP_DATA_DIR` / `CATALOG_HOME_DIR` environment variables — this notebook does
exactly that, pointing `CATALOG_HOME_DIR` at the sandbox above instead of editing the
committed file.

In [2]:
# Raw yaml — exactly what's editable in catalog_config.yaml.
config = yaml.safe_load((APP_DIR / "catalog_config.yaml").read_text())
print("id:               ", config["id"])
print("license:          ", config["license"])
print("item_folder_level:", config["item_folder_level"])
print("simulations:")
for sim in config["simulations"]:
    print(" -", sim)
print("\napp_data_dir:     ", config["app_data_dir"], " (informational only — see Step 3 below)")
print("catalog_home_dir: ", config["catalog_home_dir"], " (overridden to the sandbox for this notebook)")

# Resolved config — what stage.py/move.py actually use. HydroLand doesn't set
# property_order explicitly in the yaml, so it's worth seeing the auto-derived value.
sys.path.insert(0, str(APP_DIR / "staging"))
from stac_dedl_core.appconfig import load_app_config
resolved = load_app_config(APP_DIR)
print("\nresolved property_order:", resolved.property_order)

id:                EO.UFZ.DAT.HYDROLAND
license:           LGPL-3.0-or-later
item_folder_level: MM
simulations:
 - ifs-nemo_generation-2_historical

app_data_dir:      /data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm  (informational only — see Step 3 below)
catalog_home_dir:  /work/lealroja/applications-stac-catalog-phase3-execution  (overridden to the sandbox for this notebook)



resolved property_order: ['model', 'experiment', 'generation', 'activity', 'realization', 'data_type']


**To point this app at your own data for a real run**, edit:

- `simulations:` — which model/period combinations to publish (keys reference
  `apps/_shared/simulations.yaml`)
- `app_data_dir:` — informational only, not read by the scripts as a source path
- `catalog_home_dir:` — where the STAC catalog tree gets built (remove the
  `CATALOG_HOME_DIR` env override above to use this committed value instead of a sandbox)
- `indicators:` (optional) — decadal indicator products (aridity index, soil
  moisture, discharge percentiles), see the dedicated section near the end of this
  notebook

## 2. Step 1 — `stage.py`: build the catalog skeleton

Pure config — no NC files are read. Creates `metadata/collection.json` +
`metadata/collection_config.json`, and one `data/YYYY/MM/<item>/` folder for every
simulation × month × data_type (`mhm`/`mrm`). Dry run by default; `--execute` applies.

In [3]:
def run(*args, check=True):
    # stage.py/move.py/update.py log via Python's logging module, which writes
    # to stderr by default — print both, not just stdout, or the actual output
    # (skeleton summary, move summary, ...) goes missing.
    print("$", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=APP_DIR, env=ENV, capture_output=True, text=True)
    print(result.stdout, end="")
    print(result.stderr, end="")
    if check:
        result.check_returncode()
    return result

run(sys.executable, "staging/stage.py")  # dry run — show the plan

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/stage.py


INFO | Collection: EO.UFZ.DAT.HYDROLAND (1 simulations)
INFO | [DRY RUN] Would write collection.json + collection_config.json
INFO | [DRY RUN] Skeleton IFS-NEMO hist (1990-2014): would create 600 folders (data types: mhm, mrm)
          Example: /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_19900131T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm
INFO | [DRY RUN] /work/lealroja/ifs-nemo/5km/historic_ba/hourly/indicators/aridity/1990_01_2014_12_aridity_index.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__aridity_index/1990_01_2014_12_aridity_index.nc
INFO | [DRY RUN] /work/lealroja/ifs-nemo/5km/historic_ba/hourly/indicators/aridity_aet_x24/1990_01_2014_12_aridity_index.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_2

CompletedProcess(args=('/gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12', 'staging/stage.py'), returncode=0, stdout='', stderr="INFO | Collection: EO.UFZ.DAT.HYDROLAND (1 simulations)\nINFO | [DRY RUN] Would write collection.json + collection_config.json\nINFO | [DRY RUN] Skeleton IFS-NEMO hist (1990-2014): would create 600 folders (data types: mhm, mrm)\n          Example: /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_19900131T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm\nINFO | [DRY RUN] /work/lealroja/ifs-nemo/5km/historic_ba/hourly/indicators/aridity/1990_01_2014_12_aridity_index.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__aridity_index/1990_01_2014_12_aridity_index.nc\nINFO | [DRY RUN] /work/lealroja/ifs-nemo/5km/historic_ba/hourl

In [4]:
run(sys.executable, "staging/stage.py", "--execute")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/stage.py --execute


INFO | Collection: EO.UFZ.DAT.HYDROLAND (1 simulations)
INFO | Wrote /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/metadata/collection.json and /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/metadata/collection_config.json
INFO | Skeleton IFS-NEMO hist (1990-2014): 600 folders created
INFO | /work/lealroja/ifs-nemo/5km/historic_ba/hourly/indicators/aridity/1990_01_2014_12_aridity_index.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__aridity_index/1990_01_2014_12_aridity_index.nc
INFO | /work/lealroja/ifs-nemo/5km/historic_ba/hourly/indicators/discharge/indicators/1990_01_2014_12_1st_percentile_discharge_indicators.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__1st_percentile_discharge_indicators/1990_01_201

CompletedProcess(args=('/gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12', 'staging/stage.py', '--execute'), returncode=0, stdout='', stderr="INFO | Collection: EO.UFZ.DAT.HYDROLAND (1 simulations)\nINFO | Wrote /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/metadata/collection.json and /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/metadata/collection_config.json\nINFO | Skeleton IFS-NEMO hist (1990-2014): 600 folders created\nINFO | /work/lealroja/ifs-nemo/5km/historic_ba/hourly/indicators/aridity/1990_01_2014_12_aridity_index.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__aridity_index/1990_01_2014_12_aridity_index.nc\nINFO | /work/lealroja/ifs-nemo/5km/historic_ba/hourly/indicators/discharge/indicators/1990_01_2014_12_1st_percentile_discharge_indicators.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.D

## 3. Step 2 — `check_data.sh` (optional): completeness check

Verifies exactly one recognized monthly file exists per (year, month) in a source
directory, and flags any filename that doesn't match the recognized mHM/mRM pattern.
Run against the real historical mHM source on disk.

In [ ]:
HYDROLAND_MHM_SOURCE = "/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm"  # <-- HARDCODED PATH: change to your own mHM source folder to re-run with different data

result = subprocess.run(
    ["bash", "staging/check_data.sh", HYDROLAND_MHM_SOURCE, "1990", "2014"],
    cwd=APP_DIR, capture_output=True, text=True,
)
print(result.stdout[-2000:])  # tail — the full output lists all 300 months

## 4. Step 3 — `move.py`: bring the NC files in

`--source-flux-dir` is the usual shape: **all mHM files in one flat folder**, no
year/month subfolders — metadata comes from CLI flags, not the filenames. We link
(`--link`) rather than copy, so nothing is duplicated on disk; the sandbox just gets
symlinks pointing back at the real files.

In [6]:
move_args = [
    sys.executable, "staging/move.py",
    "--source-flux-dir", HYDROLAND_MHM_SOURCE,
    "--target-collection-dir", str(SANDBOX / COLLECTION_ID),
    "--model", "IFS-NEMO", "--experiment", "hist",
    "--generation", "2", "--activity", "baseline", "--realization", "1",
    "--link",
]
run(*move_args)  # dry run

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/move.py --source-flux-dir /data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm --target-collection-dir /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND --model IFS-NEMO --experiment hist --generation 2 --activity baseline --realization 1 --link


INFO | [symlink] /gpfs1/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm/1990_01_01_to_1990_01_31_daily_mHM_Fluxes_States.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_19900131T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm/1990_01_01_to_1990_01_31_daily_mHM_Fluxes_States.nc
INFO | [symlink] /gpfs1/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm/1990_02_01_to_1990_02_28_daily_mHM_Fluxes_States.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/02/EO.UFZ.DAT.HYDROLAND_19900201T000000_19900228T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm/1990_02_01_to_1990_02_28_daily_mHM_Fluxes_States.nc
INFO | [symlink] /gpfs1/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm/1990_03_01_to_1990_03_31_daily_mHM_Fluxes_States.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/0

CompletedProcess(args=('/gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12', 'staging/move.py', '--source-flux-dir', '/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm', '--target-collection-dir', '/tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND', '--model', 'IFS-NEMO', '--experiment', 'hist', '--generation', '2', '--activity', 'baseline', '--realization', '1', '--link'), returncode=0, stdout='', stderr='INFO | [symlink] /gpfs1/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm/1990_01_01_to_1990_01_31_daily_mHM_Fluxes_States.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_19900131T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm/1990_01_01_to_1990_01_31_daily_mHM_Fluxes_States.nc\nINFO | [symlink] /gpfs1/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm/1990_02_01_to_1990_02_28_daily_

In [7]:
run(*move_args, "--execute")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/move.py --source-flux-dir /data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm --target-collection-dir /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND --model IFS-NEMO --experiment hist --generation 2 --activity baseline --realization 1 --link --execute


INFO | [symlink] /gpfs1/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm/1990_01_01_to_1990_01_31_daily_mHM_Fluxes_States.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_19900131T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm/1990_01_01_to_1990_01_31_daily_mHM_Fluxes_States.nc
INFO | [symlink] /gpfs1/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm/1990_02_01_to_1990_02_28_daily_mHM_Fluxes_States.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/02/EO.UFZ.DAT.HYDROLAND_19900201T000000_19900228T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm/1990_02_01_to_1990_02_28_daily_mHM_Fluxes_States.nc
INFO | [symlink] /gpfs1/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm/1990_03_01_to_1990_03_31_daily_mHM_Fluxes_States.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/0

CompletedProcess(args=('/gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12', 'staging/move.py', '--source-flux-dir', '/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm', '--target-collection-dir', '/tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND', '--model', 'IFS-NEMO', '--experiment', 'hist', '--generation', '2', '--activity', 'baseline', '--realization', '1', '--link', '--execute'), returncode=0, stdout='', stderr='INFO | [symlink] /gpfs1/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm/1990_01_01_to_1990_01_31_daily_mHM_Fluxes_States.nc -> /tmp/hydroland_notebook_qy73optw/EO.UFZ.DAT.HYDROLAND/data/1990/01/EO.UFZ.DAT.HYDROLAND_19900101T000000_19900131T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm/1990_01_01_to_1990_01_31_daily_mHM_Fluxes_States.nc\nINFO | [symlink] /gpfs1/data/hsd/projects/destine_operational_results/ifs-nemo/phase_2/historical/mhm/1990_02_01_to_1990

**Two source shapes are supported** — see `--source-flux-dir` above (the usual case)
vs. `--source-item-dir` (legacy: one folder per simulation, metadata in the folder
name itself, used for migrating already-organised data). Recognized filenames in
both modes:

| Pattern | Example |
|---|---|
| `YYYY_MM_DD_to_YYYY_MM_DD_daily_mHM_Fluxes_States.nc` | `1990_01_01_to_1990_01_31_daily_mHM_Fluxes_States.nc` |
| `YYYY_MM_DD_T00_00_to_YYYY_MM_DD_T23_00_mHM_Fluxes_States.nc` | `2015_01_01_T00_00_to_2015_01_31_T23_00_mHM_Fluxes_States.nc` |
| `YYYY_MM_DD_T00_00_to_YYYY_MM_DD_T23_00_mRM_Fluxes_States.nc` | `2028_01_01_T00_00_to_2028_01_31_T23_00_mRM_Fluxes_States.nc` |

Unrecognized files are logged and skipped, never silently dropped.

## 5. Decadal indicators (optional, config-driven)

Beyond the monthly mHM/mRM fluxes, HydroLand can publish period-level **indicators**
(aridity index, discharge percentiles, soil-moisture deficit) — separate files,
staged as one item per file. `stage.py` already staged these automatically above if
your `catalog_config.yaml` has an `indicators:` block configured, e.g.:

```yaml
indicators:
  source_dir: /path/to/indicator/files
  reference_simulation: ifs-nemo_generation-2_historical   # ref / aridity / discharge
  scenario_simulation:  ifs-nemo_generation-2_projection   # scen / percent_change
  scenario_markers: [scen, percent_change]                 # token → scenario; else reference
```

The trailing `<token>` in each indicator filename (e.g. `aridity_index`,
`soil_moisture_deficit_ref_SWC_L01`) becomes the item's `data_type`. Only one
reference/scenario simulation pair is supported per `stage.py` run.

## 6. Step 4 — `update.py`: generate STAC item metadata

Writes a STAC item JSON for every folder under `data/` that has an asset, and drops
any empty skeleton folders (the ones from Step 1 that never received a file).

In [8]:
run(sys.executable, "staging/update.py")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/update.py


PySTAC version: 1.0.0
Collection passed all field checks.
item_id:EO.UFZ.DAT.HYDROLAND_20141201T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mrm, collection_id:EO.UFZ.DAT.HYDROLAND, year:2014, month:12, day:01
item_id:EO.UFZ.DAT.HYDROLAND_20141201T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm, collection_id:EO.UFZ.DAT.HYDROLAND, year:2014, month:12, day:01
item_id:EO.UFZ.DAT.HYDROLAND_20141101T000000_20141130T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mrm, collection_id:EO.UFZ.DAT.HYDROLAND, year:2014, month:11, day:01
item_id:EO.UFZ.DAT.HYDROLAND_20141101T000000_20141130T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm, collection_id:EO.UFZ.DAT.HYDROLAND, year:2014, month:11, day:01
item_id:EO.UFZ.DAT.HYDROLAND_20141001T000000_20141031T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mrm, collection_id:EO.UFZ.DAT.HYDROLAND, year:2014, month:10, day:01
item_id:EO.U

CompletedProcess(args=('/gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12', 'staging/update.py'), returncode=0, stdout='Warning: Bucket credentials not set, but IS_UPLOAD_S3 is False. Skipping upload mode.\nPySTAC version: 1.0.0\nCollection passed all field checks.\nitem_id:EO.UFZ.DAT.HYDROLAND_20141201T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mrm, collection_id:EO.UFZ.DAT.HYDROLAND, year:2014, month:12, day:01\nitem_id:EO.UFZ.DAT.HYDROLAND_20141201T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm, collection_id:EO.UFZ.DAT.HYDROLAND, year:2014, month:12, day:01\nitem_id:EO.UFZ.DAT.HYDROLAND_20141101T000000_20141130T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mrm, collection_id:EO.UFZ.DAT.HYDROLAND, year:2014, month:11, day:01\nitem_id:EO.UFZ.DAT.HYDROLAND_20141101T000000_20141130T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm, collection_id:E

## 7. Inspect the results

In [9]:
items_dir = SANDBOX / COLLECTION_ID / "metadata" / "items"
item_paths = sorted(items_dir.glob("*.json"))
print(f"{len(item_paths)} item JSON(s) generated. First 10:")
for p in item_paths[:10]:
    print(" ", p.name)

308 item JSON(s) generated. First 10:
  EO.UFZ.DAT.HYDROLAND_19900101T000000_19900131T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm.json
  EO.UFZ.DAT.HYDROLAND_19900101T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__1st_percentile_discharge_indicators.json
  EO.UFZ.DAT.HYDROLAND_19900101T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__5th_percentile_discharge_indicators.json
  EO.UFZ.DAT.HYDROLAND_19900101T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__95th_percentile_discharge_indicators.json
  EO.UFZ.DAT.HYDROLAND_19900101T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__99th_percentile_discharge_indicators.json
  EO.UFZ.DAT.HYDROLAND_19900101T000000_20141231T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__aridity_index.json
  EO.UFZ.DAT.HYDROLAND_19900101T000000_20491231T230000__IFS-NEMO__hist__generation-2__baseline__realiz

In [10]:
sample = json.loads(item_paths[0].read_text())
print("id:        ", sample["id"])
print("properties:", sample["properties"])
print("assets:    ", list(sample["assets"]))

id:         EO.UFZ.DAT.HYDROLAND_19900101T000000_19900131T230000__IFS-NEMO__hist__generation-2__baseline__realization-1__mhm
properties: {'model': 'IFS-NEMO', 'experiment': 'hist', 'generation': 'generation-2', 'activity': 'baseline', 'realization': 'realization-1', 'data_type': 'mhm', 'start_datetime': '1990-01-01T00:00:00Z', 'end_datetime': '1990-01-31T23:00:00Z', 'datetime': '1990-01-01T00:00:00Z'}
assets:     ['1990_01_01_to_1990_01_31_daily_mHM_Fluxes_States.nc']


## 8. Upload (not run in this notebook)

Skipped here on purpose — this notebook is a dry run. To actually publish:

1. Copy `apps/hydroland/.env.example` → `apps/hydroland/.env` and fill in
   `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` (see the top-level README's
   "Configure credentials" section for where these come from).
2. Remove the `CATALOG_HOME_DIR` sandbox override (run for real against the
   `catalog_home_dir` in `catalog_config.yaml`, or your own path).
3. Run:
   ```bash
   bash run_pipeline.sh --execute --upload
   bash run_pipeline.sh --execute --upload --year 1990   # resume/retry a single year
   ```

In [11]:
# Clean up the sandbox created by this notebook (symlinks only — the real source
# files on disk are never touched).
import shutil
shutil.rmtree(SANDBOX)
print(f"Removed sandbox: {SANDBOX}")

Removed sandbox: /tmp/hydroland_notebook_qy73optw
